# 🍃 Operation CropGuard — beta storyline (leaf-disease triage)

**Audience:** you, deciding whether this story is the right spine for the tutorial.
**Runs on:** CPU in about 10 minutes the first time (everything is cached afterwards). If a GPU is present, one extra section uses it.

## The mission

A farm-advisory service wants a phone app: *photograph a leaf, get a diagnosis.*
The best public training data for this is **PlantVillage**: 54,305 leaf photos, 38 crop-and-condition classes, all shot in a lab on plain backdrops.
Lab models look superb on lab photos. **Will they survive a real field?**

We test that honestly with a second dataset the model never trains on:
**RoCoLe**, 1,560 smartphone photos of *coffee* leaves taken in a real plantation in Ecuador (soil, weeds, shadows, harsh sun).

| | PlantVillage | RoCoLe |
|---|---|---|
| Role | train / validate / test the 38-class model | **evaluation only**: never trained on. Half its plants (development half) help choose the augmentation; the other half is **locked** and scored once |
| Conditions | lab, plain background | field, natural background |
| License | CC BY-SA 3.0 (Hugging Face card; some mirrors say CC0, confirm before redistributing) | CC BY 4.0 (Mendeley) |

**The catch:** coffee does not exist in PlantVillage, and RoCoLe's diseases (coffee rust, red spider mite) are not PlantVillage classes.
So the real-world check can only ask a **triage question: "healthy or sick?"** We derive it from the 38-class model as
`sick score = 1 − (sum of the probabilities of the 12 healthy classes)`.

## The lifecycle

| # | Stage | Question |
|---|---|---|
| 1 | Data and traps | What is in the data, and what will bite us? |
| 2 | Prepare | Turn raw archives into model-ready arrays |
| 3 | Baseline | How good is a lab-trained model, on lab photos and on field-like photos? |
| 4 | Shortcuts | Is the model reading leaves or backdrops? |
| 5 | Candidate augmentations | What could we do to lab photos to make them more field-like? |
| 6 | Choose a recipe | Which one helps on real photos (RoCoLe development half)? |
| 7 | Tune the head | Which classifier settings? |
| 8 | The locked real-world check | Does it hold on RoCoLe plants we never touched, with error bars? |
| 9 | (GPU only) fine-tune | Same question with a fine-tuned network |
| 10 | Package, gate, automate | How do we stop a weak model from shipping? |
| 11 | Serve | Photo in, diagnosis (or "ask an expert") out |
| 12 | Retrospective | Timings and what was painful |

**Honest caveats up front:** RoCoLe is one field, one phone, one expert's labels. Results below use **frozen** pretrained features on CPU. Absolute numbers will move when the
network is fine-tuned. The *comparison* (before vs after augmentation, same everything else, scored on photos that played no part in any choice) is the point.

In [ ]:
import os, io, re, json, time, math, base64, random, warnings, zipfile, urllib.request
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torchvision
from PIL import Image, ImageFilter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(12, os.cpu_count() or 1))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT = Path('data'); RAW = ROOT / 'raw'; CACHE = ROOT / 'cache'; ART = Path('artifacts')
for p in (RAW / 'plantvillage', RAW / 'rocole', CACHE): p.mkdir(parents=True, exist_ok=True)
PER_CLASS = 200      # PlantVillage images kept per class for this CPU-sized beta (whole leaves, so siblings stay together)
IMG = 224

timings = {}
@contextmanager
def stage(name):
    t0 = time.time(); yield
    timings[name] = time.time() - t0
    print(f'[{name}] {timings[name]:.0f}s')

print('torch', torch.__version__, '| device:', DEVICE, '| CPU threads:', torch.get_num_threads())

## 1 · The data, and what will bite us

The raw archives live under `data/raw/`. On the tutorial cluster this is the pre-loaded volume; here, if the files are missing, the next cell downloads them.

In [ ]:
PV_BASE = 'https://huggingface.co/datasets/mohanty/PlantVillage/resolve/main/'
PV_FILES = ['data.zip', 'leaf_grouping/leaf-map.json', 'splits/color_train.txt', 'splits/color_test.txt']
ROCOLE_URL = 'https://data.mendeley.com/public-api/zip/c5yvn32dzg/download/2'

def fetch(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        print('downloading', dest.name, '...'); urllib.request.urlretrieve(url, dest)

for f in PV_FILES: fetch(PV_BASE + f, RAW / 'plantvillage' / f)
fetch(ROCOLE_URL, RAW / 'rocole' / 'rocole.zip')
print({str(p.relative_to(RAW)): f'{p.stat().st_size / 1e9:.2f} GB' for p in RAW.rglob('*.zip')})

In [ ]:
# Read the file index and the leaf-grouping map that PlantVillage ships with (no images yet).
lm = json.load(open(RAW / 'plantvillage/leaf_grouping/leaf-map.json'))
rows = []
for split in ('train', 'test'):
    for p in (RAW / f'plantvillage/splits/color_{split}.txt').read_text().split('\n'):
        if not p.strip(): continue
        base = p.split('/')[-1]; key = re.sub(r'\.jpe?g$', '', base.split('___', 1)[1] if '___' in base else base, flags=re.I).lower()
        v = lm.get(key)
        rows.append({'path': p.strip(), 'provided_split': split, 'cls': p.split('/')[2], 'key': key,
                     'leaf': v[0] if v else None, 'src': key.rsplit(' ', 1)[0]})
meta = pd.DataFrame(rows)
meta['crop'] = meta.cls.str.split('___').str[0]; meta['healthy'] = meta.cls.str.endswith('healthy')
meta['gid'] = meta.leaf.where(meta.leaf.notna(), 'u' + meta.index.astype(str))     # unknown leaf -> its own group

g = meta.dropna(subset=['leaf']); both = set(g[g.provided_split == 'train'].leaf) & set(g[g.provided_split == 'test'].leaf)
purity = meta.groupby('cls').src.agg(lambda s: s.value_counts(normalize=True).iloc[0])
print(f'images: {len(meta):,} | classes: {meta.cls.nunique()} | crops: {meta.crop.nunique()} | '
      f'healthy classes: {meta[meta.healthy].cls.nunique()} ({meta.healthy.mean():.0%} of images) | diseased classes: {meta[~meta.healthy].cls.nunique()}')
print(f'images with a known leaf id: {meta.leaf.notna().mean():.1%} ({meta.leaf.nunique():,} distinct leaves, about {g.groupby("leaf").size().mean():.1f} photos each)')
print(f'leaves present in BOTH the provided train and test split: {len(both)}  ->  {g[(g.provided_split == "test") & g.leaf.isin(both)].shape[0] / (meta.provided_split == "test").sum():.1%} of test images have a sibling in train')
print(f'classes where >=90% of images come from a single capture source: {(purity >= .9).mean():.0%}')
print(f'class sizes: min {meta.cls.value_counts().min()}, median {int(meta.cls.value_counts().median())}, max {meta.cls.value_counts().max()}')
meta.groupby('crop').agg(images=('cls', 'size'), healthy_classes=('healthy', lambda s: meta.loc[s.index][s.values].cls.nunique()),
                         disease_classes=('healthy', lambda s: meta.loc[s.index][~s.values].cls.nunique())).sort_values('images', ascending=False).T

**Traps already visible before any modelling:**

1. **The shipped train/test split leaks.** The same physical leaf is photographed several times; some leaves sit on both sides of the split. We build our own split **by leaf**.
   (A quarter of the images have no leaf id, so even our split is slightly optimistic.)
2. **Class and capture session are tangled.** In most classes nearly all photos come from one collection campaign. A model can learn *the session* instead of the disease. Section 4 measures this.
3. **Some crops can't be diagnosed:** Soybean, Blueberry and Raspberry only have a healthy class; Orange and Squash only have a disease class.
4. **Imbalance:** one class has 152 images, another 5,507. Our per-class cap evens this out.

## 2 · Prepare (turn archives into model-ready arrays)

Two small, repeatable preprocessing jobs. They read straight from the zip files and write compact caches under `data/cache/`.

* **PlantVillage:** pick whole leaves up to `PER_CLASS` images per class, resize to 224 px, and extract a **leaf mask** from the matching *segmented* image.
  (The segmented files have different names than the colour ones, so they must be joined by key.)
* **RoCoLe:** read the annotation table, centre-crop each 2048×1152 photo to a square, shrink to 224 px, and keep the labels.

In [ ]:
def mask_from_segmented(seg):
    m = Image.fromarray(((np.asarray(seg).max(axis=2) > 15) * 255).astype('uint8'))
    return np.asarray(m.filter(ImageFilter.MaxFilter(5)).filter(ImageFilter.MinFilter(5))) > 127     # close small holes

def prepare_plantvillage(meta, per_class=PER_CLASS, seed=SEED):
    cache = CACHE / f'pv_{per_class}.npz'
    if cache.exists():
        z = np.load(cache, allow_pickle=True); d = {k: z[k] for k in z.files}
        d['mask'] = np.unpackbits(d.pop('mask_packed'), axis=-1).astype(bool); return d
    Z = zipfile.ZipFile(RAW / 'plantvillage/data.zip'); names = Z.namelist(); rs = random.Random(seed)
    segmap = {}
    for nm in names:
        if nm.startswith('raw/segmented/') and nm.lower().endswith('.jpg') and '___' in nm.split('/')[-1]:
            k = re.sub(r'(_final_masked)?\.jpe?g$', '', nm.split('/')[-1].split('___', 1)[1], flags=re.I).lower(); segmap[(nm.split('/')[2], k)] = nm
    picked = []
    for c, df in meta.groupby('cls'):                       # whole leaves until the class quota is reached
        gl = list(df.gid.unique()); rs.shuffle(gl); n = 0
        for gid in gl:
            sub = df[df.gid == gid]; picked.append(sub); n += len(sub)
            if n >= per_class: break
    pool = pd.concat(picked).reset_index(drop=True)
    pool = pool[[(c, k) in segmap for c, k in zip(pool.cls, pool.key)]].reset_index(drop=True)
    rd = lambda n: Image.open(io.BytesIO(Z.read(n))).convert('RGB').resize((IMG, IMG), Image.BICUBIC)
    imgs = np.zeros((len(pool), IMG, IMG, 3), 'uint8'); masks = np.zeros((len(pool), IMG, IMG), bool)
    for i, r in enumerate(pool.itertuples()):
        imgs[i] = np.asarray(rd(r.path)); masks[i] = mask_from_segmented(rd(segmap[(r.cls, r.key)]))
    out = {'img': imgs, 'cls': pool.cls.values, 'gid': pool.gid.values, 'src': pool.src.values, 'path': pool.path.values}
    np.savez(cache, mask_packed=np.packbits(masks, axis=-1), **out); out['mask'] = masks; return out

def square224(im):                                       # centre square crop, then shrink; the SAME function is reused when serving
    w, h = im.size; s = min(w, h); l, t = (w - s) // 2, (h - s) // 2
    return im.convert('RGB').crop((l, t, l + s, t + s)).resize((IMG, IMG), Image.BICUBIC)

def prepare_rocole():
    cache = CACHE / 'rocole.npz'
    if cache.exists():
        z = np.load(cache, allow_pickle=True); return {k: z[k] for k in z.files}
    Z = zipfile.ZipFile(RAW / 'rocole/rocole.zip'); B = 'RoCoLe A robusta coffee leaf images dataset/'
    ann = pd.read_csv(io.BytesIO(Z.read(B + 'Annotations/RoCoLE-csv.csv')))
    files = ann['External ID'].tolist(); cls = [json.loads(l).get('classification') for l in ann['Label']]
    imgs = np.stack([np.asarray(square224(Image.open(io.BytesIO(Z.read(B + 'Photos/' + f))))) for f in files])
    out = {'img': imgs, 'cls': np.array(cls), 'sick': np.array([c != 'healthy' for c in cls]), 'plant': np.array([f[:-6] for f in files]), 'filename': np.array(files)}
    np.savez(cache, **out); return out

with stage('2 prepare PlantVillage subset'):
    pv = prepare_plantvillage(meta)
with stage('2 prepare RoCoLe'):
    ro = prepare_rocole()
print(f"PlantVillage subset: {len(pv['img']):,} images, {len(set(pv['cls']))} classes, {len(set(pv['gid'])):,} leaf groups | arrays {pv['img'].nbytes / 1e9:.2f} GB")
print(f"RoCoLe: {len(ro['img']):,} images, {ro['plant'].size and len(set(ro['plant']))} plants | " + str(pd.Series(ro['cls']).value_counts().to_dict()))

In [ ]:
fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
rs_ = np.random.RandomState(1); pi = rs_.choice(len(pv['img']), 8, replace=False)
sick_i = rs_.choice(np.where(ro['sick'])[0], 4, replace=False); heal_i = rs_.choice(np.where(~ro['sick'])[0], 4, replace=False)
for a, i in zip(ax[0], pi): a.imshow(pv['img'][i]); a.set_title(pv['cls'][i].replace('___', '\n')[:28].replace('_', ' '), fontsize=7); a.axis('off')
for a, i in zip(ax[1], list(heal_i) + list(sick_i)): a.imshow(ro['img'][i]); a.set_title('RoCoLe: ' + str(ro['cls'][i]), fontsize=7); a.axis('off')
plt.tight_layout(); plt.show()

Top row: PlantVillage, one leaf on a plain backdrop. Bottom row: RoCoLe, real coffee leaves in a plantation. Different world, which is exactly why it is a good test.

## 3 · Baseline: a lab-trained model

We use a **frozen ResNet-18** (pretrained on ImageNet) as a feature extractor, then train a small classifier head on top. On CPU this is the practical choice;
the GPU section later fine-tunes the whole network.

We split **by leaf**, 60 / 20 / 20 into train / validation / test.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1); backbone.fc = nn.Identity()
backbone = backbone.eval().to(DEVICE)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def extract(imgs, bs=128):
    out = []
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[i:i + bs]).permute(0, 3, 1, 2).float() / 255
        out.append(backbone(((x - MEAN) / STD).to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

def cached_features(name, make_images):
    f = CACHE / f'feat_{name}.npy'
    if f.exists(): return np.load(f)
    X = extract(make_images()); np.save(f, X); return X

classes = sorted(set(pv['cls'])); cidx = {c: i for i, c in enumerate(classes)}
y = np.array([cidx[c] for c in pv['cls']]); is_healthy = np.array([c.endswith('healthy') for c in classes]); N_CLASSES = len(classes)
with stage('3 clean features (PlantVillage)'):
    X_clean = cached_features('pv_clean', lambda: pv['img'])
with stage('3 clean features (RoCoLe)'):
    X_rocole = cached_features('rocole', lambda: ro['img'])

gss = GroupShuffleSplit(1, test_size=.2, random_state=SEED); trval, test_i = next(gss.split(X_clean, y, pv['gid']))
tr_rel, va_rel = next(GroupShuffleSplit(1, test_size=.25, random_state=SEED).split(X_clean[trval], y[trval], pv['gid'][trval]))
train_i, val_i = np.random.RandomState(SEED).permutation(trval[tr_rel]), trval[va_rel]
print(f'train {len(train_i):,} | val {len(val_i):,} | test {len(test_i):,} images (split by leaf)  | shared leaves between splits: '
      f'{len(set(pv["gid"][train_i]) & set(pv["gid"][test_i]))}')
mu, sd = X_clean[train_i].mean(0), X_clean[train_i].std(0) + 1e-6      # scaling fixed from clean training features only

In [ ]:
# The leakage demo: same classifier, two ways of splitting.
def lr_acc(tr, te):
    m = LogisticRegression(max_iter=300, C=.5).fit((X_clean[tr] - mu) / sd, y[tr]); return accuracy_score(y[te], m.predict((X_clean[te] - mu) / sd))
rnd_tr, rnd_te = next(StratifiedShuffleSplit(1, test_size=.2, random_state=SEED).split(X_clean, y))
with stage('3 baseline (logistic regression, two splits)'):
    a_rnd, a_leaf = lr_acc(rnd_tr, rnd_te), lr_acc(train_i, test_i)
print(f'accuracy, random image split : {a_rnd:.3f}\naccuracy, split by leaf      : {a_leaf:.3f}   (chance: {1 / N_CLASSES:.3f})')

A few points of accuracy vanish once siblings of the same leaf can no longer sit on both sides. Splitting by leaf is the honest baseline.

## 4 · Is the model reading leaves or backdrops?

Two probes with the same classifier on a 3,000-image subset, split by leaf: **background only** (the leaf is painted over with the average backdrop colour)
and **leaf only** (backdrop blacked out).

In [ ]:
rsub = np.random.RandomState(0).choice(len(pv['img']), 3000, replace=False)
def bg_only(i):
    a = pv['img'][i].copy(); m = pv['mask'][i]; mm = np.asarray(Image.fromarray((m * 255).astype('uint8')).filter(ImageFilter.MaxFilter(7))) > 0
    a[mm] = a[~m].mean(0) if (~m).any() else a.mean((0, 1)); return a
with stage('4 shortcut probe (features for 3,000 images x 2 variants)'):
    X_bg = cached_features('pv_bgonly_3000', lambda: np.stack([bg_only(i) for i in rsub]))
    X_leaf = cached_features('pv_leafonly_3000', lambda: pv['img'][rsub] * pv['mask'][rsub][..., None])
sub_tr, sub_te = next(GroupShuffleSplit(1, test_size=.25, random_state=SEED).split(X_bg, y[rsub], pv['gid'][rsub]))
def probe(X):
    m_, s_ = X[sub_tr].mean(0), X[sub_tr].std(0) + 1e-6
    return accuracy_score(y[rsub][sub_te], LogisticRegression(max_iter=300, C=.5).fit((X[sub_tr] - m_) / s_, y[rsub][sub_tr]).predict((X[sub_te] - m_) / s_))
print(pd.Series({'chance': 1 / N_CLASSES, 'background only (no leaf)': probe(X_bg), 'leaf only (no background)': probe(X_leaf), 'whole photo': probe(X_clean[rsub])}).round(3).to_string())

With **no leaf at all**, the model still beats chance by a wide margin: the backdrop and capture session leak the label. The leaf alone carries nearly all of the real signal.
Two consequences for the rest of the notebook: we must not trust clean-photo accuracy, and training on field-like backgrounds should make the model lean on the leaf.

## 5 · Candidate augmentations

Lab photos differ from field photos in at least four ways: **background** (plain vs cluttered), **lighting and colour** (even vs harsh), **framing** (whole leaf, centred, vs close-up and partial) and **camera quality**.
Rather than guess which gap matters, we build a small menu of augmentation **recipes** and let real photos decide.

| Ingredient | What it does |
|---|---|
| `flip` | random 90° rotation and mirror |
| `rrcS` | **random resized crop**: zoom into a random region covering at least a fraction `S` of the photo, then resize back (smaller `S` = more aggressive zoom) |
| `photoS` | **photometric jitter** of strength `S`: brightness, contrast, saturation, hue, exposure, colour temperature, occasional blur |
| `synthS` | **field synthesis** of strength `S`: paste the leaf (using its mask) on a generated soil / grass / gravel background with clutter, shadow, occlusion, lighting, low resolution, JPEG |
| `bgswap` | paste the leaf on **another lab photo's backdrop**, so the backdrop stops predicting the class |

A recipe joins ingredients with `+`, applied in order: `flip+rrc0.1+photo1` means flip, then zoom (at least 10% of the area), then photometric jitter at full strength.

In [ ]:
import io
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw

PALETTES = {'soil': ((50, 34, 22), (140, 100, 66)), 'dry soil': ((110, 90, 65), (190, 165, 125)),
            'grass': ((28, 60, 20), (110, 160, 60)), 'gravel': ((80, 80, 78), (180, 175, 165)),
            'litter': ((40, 28, 18), (130, 95, 55)), 'weeds': ((35, 70, 25), (140, 170, 70))}
CLUTTER = np.array([[50, 110, 40], [90, 130, 50], [120, 100, 50], [70, 50, 30], [30, 70, 30]])

def _noise(rng, size, octaves):
    out = np.zeros((size, size), dtype=np.float32)
    for s, w in octaves:
        out += w * np.asarray(Image.fromarray((rng.random((s, s)) * 255).astype('uint8')).resize((size, size), Image.BICUBIC), dtype=np.float32) / 255
    out -= out.min()
    return out / (out.max() + 1e-6)

def _blob(size, rng, radius, blur):
    layer = Image.new('L', (size, size), 0)
    cx, cy = rng.integers(0, size, 2); a, b = radius * rng.uniform(.6, 1.4, 2)
    ImageDraw.Draw(layer).ellipse((cx - a, cy - b, cx + a, cy + b), fill=255)
    return layer.rotate(float(rng.uniform(0, 180)), center=(cx, cy)).filter(ImageFilter.GaussianBlur(blur))

def make_background(rng, size, strength):
    lo, hi = (np.array(v) for v in PALETTES[rng.choice(list(PALETTES))])
    n = _noise(rng, size, ((3, .25), (9, .25), (30, .25), (100, .15), (224, .10)))
    img = Image.fromarray((lo + n[..., None] * (hi - lo)).astype('uint8'))
    for _ in range(int(rng.integers(2, 3 + int(10 * strength)))):           # clutter: other leaves, stems, litter
        col = np.clip(CLUTTER[rng.integers(len(CLUTTER))] + rng.integers(-20, 20, 3), 0, 255)
        alpha = _blob(size, rng, size / 8, 1.2).point(lambda v: int(v * rng.uniform(.5, .9)))
        img.paste(Image.new('RGB', (size, size), tuple(int(c) for c in col)), (0, 0), alpha)
    return img

def synth_field(img, mask, seed, strength=1.0):
    """Paste a lab leaf (uint8 HxWx3 + boolean mask) into a randomised field-like photo. `strength` scales every effect (0..1)."""
    rng = np.random.default_rng(seed); size = img.shape[0]; s = float(strength)
    ys, xs = np.where(mask)
    if len(ys) < 50: ys, xs = np.array([0, size - 1]), np.array([0, size - 1])
    box = (xs.min(), ys.min(), xs.max() + 1, ys.max() + 1)
    leaf = Image.fromarray(img).crop(box); m = Image.fromarray((mask * 255).astype('uint8')).crop(box)
    ang = rng.uniform(0, 360)
    leaf, m = leaf.rotate(ang, expand=True, resample=Image.BICUBIC), m.rotate(ang, expand=True, resample=Image.BICUBIC)
    sc = rng.uniform(.45, .95) * size / max(leaf.size); new = (max(8, int(leaf.size[0] * sc)), max(8, int(leaf.size[1] * sc)))
    leaf, m = leaf.resize(new, Image.BICUBIC), m.resize(new, Image.BICUBIC).filter(ImageFilter.GaussianBlur(.8))
    ox, oy = (int(rng.uniform(-.05 * size, size - new[0] + .05 * size)), int(rng.uniform(-.05 * size, size - new[1] + .05 * size)))
    out = make_background(rng, size, s)
    sh = Image.new('L', (size, size), 0); sh.paste(m, (ox + int(rng.integers(-8, 9)), oy + int(rng.integers(-8, 9))))   # soft contact shadow
    out.paste(Image.new('RGB', (size, size), (0, 0, 0)), (0, 0), sh.filter(ImageFilter.GaussianBlur(4)).point(lambda v: int(v * rng.uniform(.3, .6))))
    out.paste(leaf, (ox, oy), m)
    a = np.asarray(out).astype('float32')
    if rng.random() < .6 * s:                                                  # occlusion by other foliage
        for _ in range(int(rng.integers(1, 3))):
            col = CLUTTER[rng.integers(len(CLUTTER))]; al = np.asarray(_blob(size, rng, size / 12, 1.0), dtype='float32')[..., None] / 255 * .9
            a = a * (1 - al) + col * al
    ramp = np.linspace(1 - .45 * s * rng.random(), 1 + .45 * s * rng.random(), size)                     # directional light
    a *= (ramp[None, :] if rng.random() < .5 else ramp[:, None])[..., None]
    if rng.random() < .5 * s:                                                  # cast shadow
        poly = Image.new('L', (size, size), 0)
        ImageDraw.Draw(poly).polygon([tuple(int(v) for v in rng.integers(0, size, 2)) for _ in range(4)], fill=255)
        a *= (1 - .55 * np.asarray(poly.filter(ImageFilter.GaussianBlur(10)), dtype='float32') / 255)[..., None]
    a *= np.array([1 + rng.uniform(-.1, .1) * s, 1, 1 + rng.uniform(-.1, .1) * s])                        # colour temperature
    out = Image.fromarray(np.clip(a, 0, 255).astype('uint8'))
    out = ImageEnhance.Brightness(out).enhance(1 + (rng.uniform(.6, 1.4) - 1) * s)
    out = ImageEnhance.Contrast(out).enhance(1 + (rng.uniform(.7, 1.3) - 1) * s)
    f = 1 - (1 - rng.uniform(.5, 1.0)) * s                                      # low-res camera
    out = out.resize((max(32, int(size * f)),) * 2, Image.BILINEAR).resize((size, size), Image.BILINEAR).filter(ImageFilter.GaussianBlur(rng.uniform(0, 1.4) * s))
    a = np.asarray(out).astype('float32') + rng.normal(0, rng.uniform(0, 7) * s, (size, size, 3))
    buf = io.BytesIO(); Image.fromarray(np.clip(a, 0, 255).astype('uint8')).save(buf, 'JPEG', quality=int(rng.integers(int(90 - 60 * s), 92)))
    return np.asarray(Image.open(io.BytesIO(buf.getvalue())).convert('RGB'))

In [ ]:
# ---- candidate augmentation regimes (all take a lab leaf: uint8 HxWx3 image + boolean leaf mask)
def _flip_rot90(a, rng):
    a = np.rot90(a, int(rng.integers(0, 4)))
    return a[:, ::-1] if rng.random() < .5 else a

def random_resized_crop(img, rng, scale=(.25, 1.0), ratio=(3 / 4, 4 / 3)):
    H = W = img.shape[0]; area = H * W * rng.uniform(*scale); ar = np.exp(rng.uniform(np.log(ratio[0]), np.log(ratio[1])))
    w, h = min(W, int(round(np.sqrt(area * ar)))), min(H, int(round(np.sqrt(area / ar))))
    x0, y0 = int(rng.integers(0, W - w + 1)), int(rng.integers(0, H - h + 1))
    return np.asarray(Image.fromarray(np.ascontiguousarray(img)).crop((x0, y0, x0 + w, y0 + h)).resize((W, H), Image.BICUBIC))

def photometric(img, rng, s=1.0):
    im = Image.fromarray(np.ascontiguousarray(img))
    im = ImageEnhance.Brightness(im).enhance(1 + rng.uniform(-.45, .45) * s); im = ImageEnhance.Contrast(im).enhance(1 + rng.uniform(-.4, .4) * s)
    im = ImageEnhance.Color(im).enhance(1 + rng.uniform(-.4, .4) * s)
    h, sat, v = im.convert('HSV').split(); h = Image.fromarray(((np.asarray(h).astype(int) + int(rng.uniform(-10, 10) * s)) % 256).astype('uint8'))
    im = Image.merge('HSV', (h, sat, v)).convert('RGB')
    a = np.asarray(im).astype('float32') / 255; a = a ** rng.uniform(1 - .35 * s, 1 + .35 * s)                       # gamma (exposure)
    a *= np.array([1 + rng.uniform(-.12, .12) * s, 1, 1 + rng.uniform(-.12, .12) * s])                                  # colour temperature
    im = Image.fromarray((np.clip(a, 0, 1) * 255).astype('uint8'))
    if rng.random() < .5 * s: im = im.filter(ImageFilter.GaussianBlur(rng.uniform(.3, 1.3)))
    return np.asarray(im)

def bg_swap(img, mask, other_img, other_mask, rng):
    """Put the leaf on another photo's lab backdrop (its leaf painted out), so the backdrop no longer tells you the class."""
    bgm = np.asarray(Image.fromarray((other_mask * 255).astype('uint8')).filter(ImageFilter.MaxFilter(9))) > 0
    bg = other_img.copy(); bg[bgm] = bg[~bgm].mean(0) if (~bgm).any() else bg.mean((0, 1))
    size = img.shape[0]; ys, xs = np.where(mask)
    if len(ys) < 50: return _flip_rot90(img, rng)
    box = (xs.min(), ys.min(), xs.max() + 1, ys.max() + 1)
    leaf = Image.fromarray(img).crop(box); m = Image.fromarray((mask * 255).astype('uint8')).crop(box)
    ang = rng.uniform(0, 360); leaf, m = leaf.rotate(ang, expand=True, resample=Image.BICUBIC), m.rotate(ang, expand=True, resample=Image.BICUBIC)
    sc = rng.uniform(.55, .98) * size / max(leaf.size); new = (max(8, int(leaf.size[0] * sc)), max(8, int(leaf.size[1] * sc)))
    leaf, m = leaf.resize(new, Image.BICUBIC), m.resize(new, Image.BICUBIC).filter(ImageFilter.GaussianBlur(.8))
    out = Image.fromarray(bg); out.paste(leaf, (int(rng.uniform(0, size - new[0])), int(rng.uniform(0, size - new[1]))), m); return np.asarray(out)

LEGACY = {'photo': 'flip+photo1', 'rrc': 'flip+rrc0.25+photo1', 'bgswap': 'bgswap', 'bgswap+rrc': 'bgswap+rrc0.25+photo1'}

def make_regime(name, i, idx_pool, rng_seed):
    """One augmented copy (uint8 224x224x3) of training image i. `name` is a recipe of ops joined by '+':
    flip (rot90/mirror) | rrcS (random resized crop, min area S) | photoS (photometric jitter, strength S) | bgswapX | synthS (field synthesis, strength S)."""
    rng = np.random.default_rng(rng_seed); img, mask = pv['img'][i], pv['mask'][i]
    for op in LEGACY.get(name, name).split('+'):
        if op == 'none': continue
        if op == 'flip': img = _flip_rot90(img, rng)
        elif op.startswith('rrc'): img = random_resized_crop(img, rng, scale=(float(op[3:]), 1.0))
        elif op.startswith('photo'): img = photometric(img, rng, float(op[5:])) if float(op[5:]) > 0 else img
        elif op.startswith('synth'): img = synth_field(img, mask, rng_seed, float(op[5:]))
        elif op.startswith('bgswap'):
            j = int(idx_pool[rng.integers(len(idx_pool))]); img = bg_swap(img, mask, pv['img'][j], pv['mask'][j], rng)
        else: raise ValueError(op)
    return img

In [ ]:
def regime_images(spec, idx, seed_base):
    return np.stack([make_regime(spec, int(i), train_i, seed_base + int(i)) for i in idx])

SHOW = ['none', 'synth0.5', 'synth1.0', 'flip+photo1', 'flip+rrc0.25+photo1', 'flip+rrc0.1+photo1', 'bgswap+rrc0.25+photo1']
demo = np.random.RandomState(3).choice(train_i, 3, replace=False)
fig, ax = plt.subplots(3, len(SHOW), figsize=(2.3 * len(SHOW), 7.2))
for r, i in enumerate(demo):
    for c, spec in enumerate(SHOW):
        ax[r, c].imshow(make_regime(spec, int(i), train_i, 11 + int(i))); ax[r, c].axis('off')
        if r == 0: ax[r, c].set_title(spec, fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Feature sets for the two synthetic-field recipes (and a harsh synthetic test set, used only as a stress reference).
field_images = lambda idx, strength, base_seed: np.stack([synth_field(pv['img'][i], pv['mask'][i], base_seed + int(i), strength) for i in idx])
with stage('5 synthetic-field feature sets (train copies x2 strengths, stress test)'):
    A05 = cached_features('aug_train_s05', lambda: field_images(train_i, .5, 100_000))
    A10 = cached_features('aug_train_s10', lambda: field_images(train_i, 1.0, 200_000))
    F_test = cached_features('field_test', lambda: field_images(test_i, 1.0, 400_000))

## 6 · Which recipe actually helps on real photos?

We need a real-world signal to choose with, and we must not contaminate the final score. So **RoCoLe is split by plant into two halves**:

* **development half** (195 plants): used to choose the augmentation recipe and the head settings;
* **locked half** (195 plants): scored **once, at the end**, after every choice is made.

The earlier synthetic-field accuracy turned out *not* to be a reliable guide, so the sweep below is judged on the development half. For each recipe we train the head on clean data plus one augmented copy of the training set
and measure the **healthy-vs-sick ranking (AUC)** on the development plants, twice: with the 38-class model's *derived* triage score (our product), and with a *direct* healthy-vs-sick head (a reference).
We also track clean PlantVillage validation accuracy to see what each recipe costs.

In [ ]:
class Head(nn.Module):
    def __init__(self, hidden=0, dropout=0.0, n_out=None):
        super().__init__(); n_out = n_out or N_CLASSES
        self.net = nn.Linear(512, n_out) if hidden == 0 else nn.Sequential(nn.Linear(512, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, n_out))
    def forward(self, x): return self.net(x)

std = lambda X: torch.tensor((X - mu) / sd, dtype=torch.float32)

def fit_head(cfg, seed=SEED, Xa=None, binary=False, epochs=30, batch=256):
    # Xa: features of one augmented copy of the training set (same row order as train_i); cfg['aug_fraction'] says how much of it to use.
    k = int(cfg.get('aug_fraction', 1.0) * len(train_i)) if Xa is not None else 0
    X = np.vstack([X_clean[train_i], Xa[:k]]) if k else X_clean[train_i]; yy = np.concatenate([y[train_i], y[train_i][:k]])
    if binary: yy = (~is_healthy[yy]).astype(int)                      # reference: train healthy-vs-sick directly
    torch.manual_seed(seed); X, yy = std(X), torch.tensor(yy)
    head = Head(cfg['hidden'], cfg['dropout'], 2 if binary else None); opt = torch.optim.Adam(head.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    lossf = nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    for _ in range(epochs):
        head.train(); order = torch.randperm(len(X))
        for i in range(0, len(order), batch):
            j = order[i:i + batch]; opt.zero_grad(); lossf(head(X[j]), yy[j]).backward(); opt.step()
    return head.eval()

def proba(head, X):
    with torch.no_grad(): return torch.softmax(head(std(X)), 1).numpy()

def triage_score(P): return 1 - P[:, is_healthy].sum(1)                  # P(sick) derived from the 38-class output

# RoCoLe: split plants into a development half and a locked half
y_sick_ro = ro['sick'].astype(int); plants = ro['plant']
up = np.unique(plants); shuffled = np.random.RandomState(123).permutation(up); dev_plants = set(shuffled[:len(up) // 2])
is_dev = np.array([p in dev_plants for p in plants]); is_locked = ~is_dev
print(f'RoCoLe development half: {is_dev.sum()} photos / {len(dev_plants)} plants | locked half: {is_locked.sum()} photos / {len(up) - len(dev_plants)} plants')
def ro_metrics(s, mask): return {'auc': roc_auc_score(y_sick_ro[mask], s[mask]), 'bal_acc': balanced_accuracy_score(y_sick_ro[mask], s[mask] >= .5)}

In [ ]:
REGIMES = ['none', 'synth0.5', 'synth1.0', 'flip+photo1', 'flip+rrc0.25', 'flip+rrc0.25+photo1', 'flip+rrc0.5+photo1', 'flip+rrc0.1+photo1', 'bgswap+rrc0.25+photo1', 'synth0.35+rrc0.35+photo0.6']
CACHED = {'synth0.5': lambda: A05, 'synth1.0': lambda: A10}
def recipe_features(spec, k):
    if spec == 'none': return None
    if spec in CACHED: return CACHED[spec]()
    return cached_features('reg_' + spec.replace('+', '_'), lambda: regime_images(spec, train_i, 500_000 + 100_000 * k))

SWEEP_CFG = dict(hidden=512, dropout=0.5, lr=1e-3, wd=1e-5, label_smoothing=0.1)
def eval_recipe(Xa, mask, seeds=3, cfg=SWEEP_CFG):
    rows = {'der_auc': [], 'der_bal': [], 'bin_auc': [], 'bin_bal': [], 'pv_val_acc': []}
    for s in range(seeds):
        hd, hb = fit_head(cfg, SEED + 10 * s, Xa), fit_head(cfg, SEED + 10 * s, Xa, binary=True)
        m1, m2 = ro_metrics(triage_score(proba(hd, X_rocole)), mask), ro_metrics(proba(hb, X_rocole)[:, 1], mask)
        rows['der_auc'].append(m1['auc']); rows['der_bal'].append(m1['bal_acc']); rows['bin_auc'].append(m2['auc']); rows['bin_bal'].append(m2['bal_acc'])
        rows['pv_val_acc'].append(accuracy_score(y[val_i], proba(hd, X_clean[val_i]).argmax(1)))
    return {k: float(np.mean(v)) for k, v in rows.items()}

with stage('6 recipe sweep on the RoCoLe development half (10 recipes x 6 heads)'):
    sweep = pd.DataFrame({spec: eval_recipe(recipe_features(spec, k), is_dev) for k, spec in enumerate(REGIMES)}).T
sweep.columns = ['dev derived AUC', 'dev derived bal acc', 'dev direct AUC', 'dev direct bal acc', 'PV clean val acc']
sweep.sort_values('dev derived AUC', ascending=False).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2)); order = sweep.sort_values('dev derived AUC').index; yy_ = np.arange(len(order))
ax.barh(yy_ + .2, sweep.loc[order, 'dev derived AUC'], height=.4, label='derived from 38 classes (our product)', color='#2a6fb0')
ax.barh(yy_ - .2, sweep.loc[order, 'dev direct AUC'], height=.4, label='direct healthy-vs-sick head (reference)', color='#9bb7d4')
ax.axvline(sweep.loc['none', 'dev derived AUC'], color='#2a6fb0', ls=':', lw=1); ax.axvline(sweep.loc['none', 'dev direct AUC'], color='#7a9cc0', ls=':', lw=1)
ax.set_yticks(yy_, order, fontsize=8); ax.set_xlim(.5, .85); ax.set_xlabel('AUC on RoCoLe development plants (dotted: clean-only)'); ax.legend(frameon=False, fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

**The selection rule is fixed in code before anything touches the locked half:** among recipes that keep clean validation accuracy within 5 points of clean-only **and** do not hurt the direct reference, pick the one with the highest
development AUC for the derived triage score (our product).

In [ ]:
base_row = sweep.loc['none']
eligible = sweep.drop(index='none')
eligible = eligible[(eligible['PV clean val acc'] >= base_row['PV clean val acc'] - .05) & (eligible['dev direct AUC'] >= base_row['dev direct AUC'])]
BEST = eligible['dev derived AUC'].idxmax(); Xa_best = recipe_features(BEST, REGIMES.index(BEST))
print('eligible recipes:', list(eligible.index)); print('chosen recipe  :', BEST)
print(sweep.loc[['none', BEST]].round(3).to_string())

Read the sweep with some care: with 195 development plants, differences of a couple of points between neighbouring recipes are within noise (repeat draws of the same recipe moved AUC by about ±0.01 in my testing), so do not over-read the exact winner.
The larger, repeatable patterns are what matter:

* **Plain framing and colour augmentation (zoom crops plus photometric jitter) beat the elaborate synthetic-field backgrounds.** Synthetic fields helped the derived score somewhat but did nothing for the direct head and cost the most clean accuracy.
* **Photometric jitter matters for the derived score;** crops alone were noticeably weaker there.
* **Adding a backdrop swap on top of crops and jitter did not help** (compare `bgswap+rrc0.25+photo1` with `flip+rrc0.25+photo1`).

## 7 · Tune the head (with the chosen recipe)

Now the recipe is fixed, we tune the classifier head: width (0 = plain linear), dropout, learning rate, weight decay, label smoothing, and **how much of the augmented copy to use** (half or all).
Each **trial** is one training run. The objective is development-half derived AUC, **and a trial only counts if clean validation accuracy stays within 5 points of clean-only**. The locked half is still untouched.

In [ ]:
N_TRIALS = 16; space = np.random.RandomState(SEED); base_clean = float(sweep.loc['none', 'PV clean val acc'])
def sample_cfg():
    return dict(hidden=int(space.choice([0, 256, 512])), dropout=float(space.choice([0.0, 0.2, 0.5])), lr=float(10 ** space.uniform(-3.3, -2)),
                wd=float(10 ** space.uniform(-6, -3)), label_smoothing=float(space.choice([0.0, 0.1])), aug_fraction=float(space.choice([0.5, 1.0])))
trials = []
with stage('7 head tuning (16 trials, one after another)'):
    for t in range(N_TRIALS):
        cfg = sample_cfg(); t0 = time.time(); h = fit_head(cfg, SEED + t, Xa_best)
        c = accuracy_score(y[val_i], proba(h, X_clean[val_i]).argmax(1)); a = ro_metrics(triage_score(proba(h, X_rocole)), is_dev)['auc']
        trials.append({'trial': t, **cfg, 'clean_val_acc': c, 'dev_auc': a, 'objective': a if c >= base_clean - .05 else -1, 'seconds': time.time() - t0})
        print(f"trial {t:>2}: hidden {cfg['hidden']:>3} drop {cfg['dropout']:.1f} lr {cfg['lr']:.4f} aug {cfg['aug_fraction']:.1f}  ->  clean val {c:.3f}  dev AUC {a:.3f}  ({trials[-1]['seconds']:.0f}s)")
results = pd.DataFrame(trials).sort_values('objective', ascending=False).reset_index(drop=True)
best_cfg = {k: (int(v) if k == 'hidden' else float(v)) for k, v in results.loc[0, ['hidden', 'dropout', 'lr', 'wd', 'label_smoothing', 'aug_fraction']].items()}
print('\nbest settings:', best_cfg)
results.round(3).head(6)

## 8 · The locked real-world check

Two models, **identical in every setting except the augmented data**:

* **before:** the best head settings, clean PlantVillage only
* **after:** the same settings plus the chosen augmentation recipe

Three seeds each. Scored on clean PlantVillage test, a *harsh synthetic* field test (context only), and, for the first time, the **locked half of RoCoLe** (195 plants never used for any choice).
Error bars come from a **bootstrap over those plants**, since RoCoLe has four photos per plant.

In [ ]:
cfg_before = {**best_cfg, 'aug_fraction': 0.0}; cfg_after = dict(best_cfg)
runs = {}
with stage('8 final training (2 configs x 3 seeds)'):
    for name, cfg, Xa in (('before (clean only)', cfg_before, None), ('after (+ augmentation)', cfg_after, Xa_best)):
        heads = [fit_head(cfg, SEED + 100 + s, Xa) for s in range(3)]
        runs[name] = {'heads': heads,
                      'clean': [{'acc': accuracy_score(y[test_i], proba(h, X_clean[test_i]).argmax(1))} for h in heads],
                      'field': [{'acc': accuracy_score(y[test_i], proba(h, F_test).argmax(1))} for h in heads],
                      'rocole_scores': np.mean([triage_score(proba(h, X_rocole)) for h in heads], axis=0),
                      'rocole': [ro_metrics(triage_score(proba(h, X_rocole)), is_locked) for h in heads]}
agg = lambda lst, k: f"{np.mean([d[k] for d in lst]):.3f} ± {np.std([d[k] for d in lst]):.3f}"
table = pd.DataFrame({n: {'PlantVillage clean test: 38-class accuracy': agg(r['clean'], 'acc'),
                          'PlantVillage harsh synthetic-field test: 38-class accuracy (context only)': agg(r['field'], 'acc'),
                          'RoCoLe LOCKED half: derived triage AUC': agg(r['rocole'], 'auc'),
                          'RoCoLe LOCKED half: derived triage balanced accuracy @0.5': agg(r['rocole'], 'bal_acc'),
                          'RoCoLe LOCKED half: share predicted sick (true share %.2f)' % y_sick_ro[is_locked].mean(): f"{np.mean([(triage_score(proba(h, X_rocole))[is_locked] >= .5).mean() for h in r['heads']]):.2f}"}
                      for n, r in runs.items()})
table

In [ ]:
def ro_metrics_idx(s, idx): return {'auc': roc_auc_score(y_sick_ro[idx], s[idx]), 'bal_acc': balanced_accuracy_score(y_sick_ro[idx], s[idx] >= .5)}

# Reference: a healthy-vs-sick head trained directly, same features and settings, scored on the same locked plants.
ref_rows = {}
for name, cfg, Xa in (('before (clean only)', cfg_before, None), ('after (+ augmentation)', cfg_after, Xa_best)):
    ms = [ro_metrics(proba(fit_head(cfg, SEED + 200 + s, Xa, binary=True), X_rocole)[:, 1], is_locked) for s in range(3)]
    ref_rows[name] = {'RoCoLe LOCKED: direct triage AUC': agg(ms, 'auc'), 'RoCoLe LOCKED: direct triage balanced accuracy @0.5': agg(ms, 'bal_acc')}
print('direct healthy-vs-sick head (reference):'); print(pd.DataFrame(ref_rows).to_string())

# Bootstrap over the locked plants: how much of the before -> after change is real?
rng = np.random.RandomState(0); lp = np.unique(plants[is_locked]); by_plant = {p: np.where((plants == p) & is_locked)[0] for p in lp}
sb, sa = runs['before (clean only)']['rocole_scores'], runs['after (+ augmentation)']['rocole_scores']
d_auc, d_bal = [], []
for _ in range(1000):
    idx = np.concatenate([by_plant[p] for p in rng.choice(lp, len(lp))]); ma, mb = ro_metrics_idx(sa, idx), ro_metrics_idx(sb, idx)
    d_auc.append(ma['auc'] - mb['auc']); d_bal.append(ma['bal_acc'] - mb['bal_acc'])
ci = lambda v: f'{np.mean(v):+.3f}  (95% CI {np.percentile(v, 2.5):+.3f} to {np.percentile(v, 97.5):+.3f})'
print('\nRoCoLe locked half, after minus before, bootstrap over plants:\n  derived triage AUC           ', ci(d_auc), '\n  derived triage balanced acc  ', ci(d_bal))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (name, r) in zip(ax, runs.items()):
    s_ = r['rocole_scores']; a.hist(s_[is_locked & ~ro['sick']], bins=25, alpha=.65, label='healthy', color='#2a9d5c'); a.hist(s_[is_locked & ro['sick']], bins=25, alpha=.65, label='sick', color='#d9625a')
    a.axvline(.5, color='k', lw=1, ls=':'); a.set_title(name + ': locked-half sick scores'); a.set_xlabel('derived sick score'); a.legend(frameon=False)
plt.tight_layout(); plt.show()

**Reading this honestly:**

* The recipe was chosen on the **development** plants, and this is the **locked** half. If the gain here matches the development gain, it is real rather than an artefact of picking the winner from a menu.
* The clean-only model calls almost every coffee leaf *sick* (compare its share predicted sick with the true ~49%). With the recipe the scores spread out and the predicted share moves toward the truth, but it is still biased toward *sick* (look at the table: roughly three in four coffee leaves are still called sick, against about half in reality).
* The **direct** healthy-vs-sick reference is the fair yardstick for "is derived triage good enough?". Compare the two tables: the recipe also helps the direct head, by a smaller margin, and with the recipe the derived score ranks about as well as the direct head does.
* The bootstrap covers sampling noise over plants. It does **not** cover the fact that this is **one field, one phone and one annotator**, nor that everything here uses frozen features. Whether the gain carries over to a *fine-tuned* network is exactly what section 9 is for.
* The synthetic-field row is context only: a recipe built to look like the field is not the same thing as one that helps on the real field.

## 9 · (GPU only) fine-tune the whole network

Everything above used frozen features so it runs anywhere. With a GPU the same question can be asked of a **fine-tuned ResNet-18** trained on pixels:
clean-only vs clean plus one augmented copy under the chosen recipe, scored on clean PlantVillage test and on the **locked** RoCoLe half. This cell is skipped without a GPU.

In [ ]:
def to_batch(arr, device):
    x = torch.from_numpy(arr).to(device).permute(0, 3, 1, 2).float() / 255
    return (x - MEAN.to(device)) / STD.to(device)

def finetune(train_imgs, train_y, epochs=4, lr=3e-4, batch=64, device=DEVICE, max_steps=None, seed=SEED):
    torch.manual_seed(seed); net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    net.fc = nn.Linear(512, N_CLASSES); net.to(device)
    steps = epochs * math.ceil(len(train_imgs) / batch); opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps); amp = device == 'cuda'
    scaler = torch.amp.GradScaler(enabled=amp); yt = torch.tensor(train_y); n = 0
    for ep in range(epochs):
        net.train(); order = torch.randperm(len(train_imgs))
        for i in range(0, len(order), batch):
            j = order[i:i + batch]; x = to_batch(train_imgs[j.numpy()], device)
            flip = torch.rand(len(j), device=device) < .5; x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
            opt.zero_grad()
            with torch.autocast(device_type=device, enabled=amp): loss = nn.functional.cross_entropy(net(x), yt[j].to(device))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); n += 1
            if max_steps and n >= max_steps: return net.eval()
    return net.eval()

@torch.no_grad()
def net_proba(net, imgs, device=DEVICE, bs=128):
    return np.concatenate([torch.softmax(net(to_batch(imgs[i:i + bs], device)), 1).cpu().numpy() for i in range(0, len(imgs), bs)])

if DEVICE == 'cuda':
    with stage('9 GPU fine-tune: before and after augmentation'):
        aug_imgs = regime_images(BEST, train_i, 700_000)
        nets = {'before (clean only)': finetune(pv['img'][train_i], y[train_i]),
                'after (+ augmentation)': finetune(np.concatenate([pv['img'][train_i], aug_imgs]), np.concatenate([y[train_i], y[train_i]]))}
        ft = {}
        for name, net in nets.items():
            P_ro = net_proba(net, ro['img']); P_c = net_proba(net, pv['img'][test_i]); m_ = ro_metrics(triage_score(P_ro), is_locked)
            ft[name] = {'PV clean test acc': accuracy_score(y[test_i], P_c.argmax(1)), 'RoCoLe locked: derived triage AUC': m_['auc'], 'RoCoLe locked: derived triage bal acc': m_['bal_acc']}
    display(pd.DataFrame(ft).round(3))
else:
    print('no GPU in this environment: skipped (the cell defines finetune() and net_proba() so it can be run elsewhere).')

## 10 · Package, gate, automate

A saved model needs more than weights: the **class list, which classes are healthy, the scaling statistics, the preprocessing recipe, the decision thresholds**, and
a **reference set of training features** so the service can say *"this photo looks like nothing I was trained on"*.

The gate is the automation: given a trained model it computes the same report every time and refuses to ship if anything is below the bar.

In [ ]:
# Reference features and an "unfamiliar photo" distance: mean cosine distance to the 5 nearest reference features.
ref_pool = np.vstack([X_clean[train_i], Xa_best]); ref = ref_pool[np.random.RandomState(0).choice(len(ref_pool), 3000, replace=False)]
unit = lambda X: X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def knn_distance(X, ref_):
    S = unit(X) @ unit(ref_).T; return 1 - np.sort(S, axis=1)[:, -5:].mean(1)
X_val_aug = cached_features('val_aug_' + BEST.replace('+', '_'), lambda: regime_images(BEST, val_i, 800_000))
d_val = np.concatenate([knn_distance(X_clean[val_i], ref), knn_distance(X_val_aug, ref)])
OOD_THRESHOLD = float(np.percentile(d_val, 99))           # abstain on anything farther than 99% of known-good validation photos
print(f'unfamiliar-photo threshold (mean 5-NN cosine distance): {OOD_THRESHOLD:.3f}')

from sklearn.datasets import load_sample_images                 # two real photos that ship with scikit-learn: a temple and a flower
photos = load_sample_images().images; crng = np.random.RandomState(0)
def crops(n):
    out = []
    for k in range(n):
        im = photos[k % 2]; y0, x0 = crng.randint(0, im.shape[0] - 300), crng.randint(0, im.shape[1] - 300)
        out.append(np.asarray(Image.fromarray(im[y0:y0 + 300, x0:x0 + 300]).resize((IMG, IMG))))
    return np.stack(out)
non_leaf = np.concatenate([crops(150), (crng.rand(50, IMG, IMG, 3) * 255).astype('uint8')])        # real non-leaf crops + pure noise
sets_for_ood = {'PV clean test': X_clean[test_i], 'PV harsh synthetic-field test': F_test, 'RoCoLe coffee (real field)': X_rocole, 'not leaves (temple/flower crops + noise)': extract(non_leaf)}
print(pd.Series({k: f'{(knn_distance(v, ref) > OOD_THRESHOLD).mean():.0%}' for k, v in sets_for_ood.items()}, name='share flagged "unfamiliar"').to_string())

The familiarity check lets clean photos through and flags non-leaf images. It flags only **about half of the coffee photos**: the augmented reference set covers more of feature space, so an unknown crop looks less unfamiliar than it would with a clean-only reference.
That is a real trade-off of augmenting (more robust, but a weaker crop-awareness check), and the unfamiliar threshold would need tuning for a real release. (The non-leaf set is only two photos' crops plus noise, so treat that row as a smoke test, not a benchmark.)

In [ ]:
final_head = runs['after (+ augmentation)']['heads'][0]
CONF_THRESHOLD = 0.5                                         # below this top-class probability the app says "ask an expert"

def selective_report(P, idx):
    conf = P.max(1); rows = {}
    for t in (0.0, 0.5, 0.7, 0.9):
        keep = conf >= t; rows[f'confidence >= {t}'] = {'coverage': keep.mean(), 'accuracy on answered': accuracy_score(y[idx][keep], P.argmax(1)[keep]) if keep.any() else np.nan}
    return pd.DataFrame(rows).T
print('PV clean test, answer only when confident:'); print(selective_report(proba(final_head, X_clean[test_i]), test_i).round(3).to_string())

def save_artifact(head, version, report):
    path = ART / f'cropguard_{version}'; path.mkdir(parents=True, exist_ok=True)
    torch.save({'state_dict': head.state_dict(), 'hidden': best_cfg['hidden'], 'dropout': best_cfg['dropout']}, path / 'head.pt')
    np.save(path / 'reference_features.npy', ref.astype('float16'))
    meta_ = {'name': 'cropguard_leaf_triage', 'version': version, 'created': pd.Timestamp.now().isoformat(timespec='seconds'),
             'backbone': 'torchvision resnet18, IMAGENET1K_V1, frozen, 224x224, ImageNet normalisation', 'preprocess': 'centre square crop, resize to 224',
             'augmentation_recipe': BEST, 'classes': classes, 'healthy_class_ids': [int(i) for i in np.where(is_healthy)[0]], 'mean': mu.tolist(), 'std': sd.tolist(),
             'confidence_threshold': CONF_THRESHOLD, 'unfamiliar_threshold': OOD_THRESHOLD, 'hyperparameters': best_cfg, 'metrics': report,
             'data': {'train': 'PlantVillage (CC BY-SA 3.0 per Hugging Face card), %d images, split by leaf' % len(train_i),
                      'real-world check': 'RoCoLe (CC BY 4.0): Parraga-Alava et al., Data in Brief 2019, doi 10.17632/c5yvn32dzg.2. Never trained on. Half the plants chose the recipe and head settings; the other half was scored once.'},
             'known_limits': ['trained on lab photos plus augmentation, not real field photos', 'coffee and any crop outside the 14 PlantVillage crops is out of scope',
                              'real-world triage was checked on a single coffee plantation in Ecuador, with frozen features']}
    (path / 'metadata.json').write_text(json.dumps(meta_, indent=1)); return path

GATE = {'max_clean_acc_drop': 0.05, 'min_clean_acc': 0.85}
READINESS_TARGET = {'rocole_triage_auc': 0.80}                 # illustrative bar for a real release, NOT used to block the beta
mean = lambda r, k, m: float(np.mean([d[m] for d in r[k]]))

def deployment_gate(challenger='after (+ augmentation)', champion='before (clean only)'):
    c, k = runs[challenger], runs[champion]; reasons = []
    if mean(k, 'clean', 'acc') - mean(c, 'clean', 'acc') > GATE['max_clean_acc_drop']: reasons.append('lost too much clean accuracy vs the current model')
    if mean(c, 'clean', 'acc') < GATE['min_clean_acc']: reasons.append('clean accuracy below the floor')
    if challenger == champion or np.percentile(d_auc, 2.5) <= 0: reasons.append('no statistically clear gain on the real-world canary (locked RoCoLe plants)')
    return {'candidate': challenger, 'passed': not reasons, 'reasons': reasons, 'clean_acc': round(mean(c, 'clean', 'acc'), 3),
            'rocole_locked_auc': round(mean(c, 'rocole', 'auc'), 3), 'rocole_auc_gain_ci_low': round(float(np.percentile(d_auc, 2.5)), 3)}

print('champion = clean-only model; challenger = augmented model\nbars:', GATE)
verdict = deployment_gate(); print(verdict)
ready = verdict['rocole_locked_auc'] >= READINESS_TARGET['rocole_triage_auc']
print(f"\nproduction readiness (real-world triage AUC >= {READINESS_TARGET['rocole_triage_auc']}): {'MET' if ready else 'NOT MET'} ({verdict['rocole_locked_auc']})")
print('\nsanity check, clean-only model as its own challenger ->', deployment_gate('before (clean only)', 'before (clean only)')['reasons'])

report = {'clean_acc': mean(runs['after (+ augmentation)'], 'clean', 'acc'), 'rocole_locked_triage_auc': mean(runs['after (+ augmentation)'], 'rocole', 'auc'),
          'rocole_locked_triage_bal_acc': mean(runs['after (+ augmentation)'], 'rocole', 'bal_acc'), 'rocole_auc_gain_vs_clean_only_ci95': [float(np.percentile(d_auc, 2.5)), float(np.percentile(d_auc, 97.5))]}
report['status'] = 'beta: passed the regression gate; real-world readiness target ' + ('met' if ready else 'NOT met')
art = save_artifact(final_head, 'v1', report) if verdict['passed'] else None
print('\nartifact:', art, {f.name: f'{f.stat().st_size / 1024:.0f} KB' for f in art.iterdir()} if art else '(gate blocked: nothing saved)')

This is a **champion / challenger** gate: a new model ships only if it does not regress the current one on clean photos **and** shows a statistically clear gain on the real-world canary (the locked RoCoLe plants).
The bars are policy choices; a team would agree them up front.

Passing a regression gate is not the same as being ready: the readiness line shows how far the model is from an **illustrative production target**, which is why the artifact is labelled *beta*.
The natural next steps are fine-tuning on a GPU (section 9) and adding real field photos to training.

## 11 · Serve it

`CropGuardService` is built **only from the saved files**. It takes a photo, applies the same preprocessing as training, and answers in one of two ways:
a **diagnosis** (top 3 conditions plus the healthy/sick score) or **"refer to an extension officer"** when the model is unconfident or the photo looks unlike anything it was trained on.

In [ ]:
class CropGuardService:
    def __init__(self, path):
        path = Path(path); self.meta = json.loads((path / 'metadata.json').read_text()); ck = torch.load(path / 'head.pt', weights_only=True)
        self.head = Head(ck['hidden'], ck['dropout']); self.head.load_state_dict(ck['state_dict']); self.head.eval()
        self.ref = np.load(path / 'reference_features.npy').astype('float32'); self.mu, self.sd = np.array(self.meta['mean']), np.array(self.meta['std'])
        self.healthy = np.array(self.meta['healthy_class_ids'])

    def predict(self, payload):
        out = []
        for inst in payload['instances']:
            try: im = Image.open(io.BytesIO(base64.b64decode(inst['image_b64'])))
            except Exception: out.append({'error': 'could not decode image'}); continue
            x = extract(np.asarray(square224(im))[None]); dist = float(knn_distance(x, self.ref)[0])
            with torch.no_grad(): p = torch.softmax(self.head(torch.tensor((x - self.mu) / self.sd, dtype=torch.float32)), 1).numpy()[0]
            top = np.argsort(p)[::-1][:3]; rec = {'top3': [{'condition': self.meta['classes'][i], 'p': round(float(p[i]), 3)} for i in top], 'p_sick': round(float(1 - p[self.healthy].sum()), 3), 'unfamiliar_distance': round(dist, 3)}
            if dist > self.meta['unfamiliar_threshold']: rec['decision'] = 'refer to extension officer (photo looks unlike the training data)'
            elif p.max() < self.meta['confidence_threshold']: rec['decision'] = 'refer to extension officer (low confidence)'
            else: rec['decision'] = 'diagnosis'
            out.append(rec)
        return {'model': f"{self.meta['name']}:{self.meta['version']}", 'predictions': out}

def b64jpeg(arr):
    b = io.BytesIO(); Image.fromarray(arr).save(b, 'JPEG', quality=92); return base64.b64encode(b.getvalue()).decode()

service = CropGuardService(art)
ti = test_i[0]; sick_ro, heal_ro = int(np.where(ro['sick'])[0][0]), int(np.where(~ro['sick'])[0][0])
cases = {'PlantVillage lab leaf (' + pv['cls'][ti].replace('___', ' / ') + ')': pv['img'][ti],
         'same leaf, harsh synthetic field photo': synth_field(pv['img'][ti], pv['mask'][ti], 7, 1.0),
         'RoCoLe coffee leaf, healthy': ro['img'][heal_ro], 'RoCoLe coffee leaf, rust/mite': ro['img'][sick_ro],
         'not a leaf (temple/flower crop)': non_leaf[0]}
truths = [pv['cls'][ti], pv['cls'][ti] + ' (same leaf)', 'healthy coffee', 'coffee ' + str(ro['cls'][sick_ro]), 'not a leaf']
resp = service.predict({'instances': [{'image_b64': b64jpeg(a)} for a in cases.values()]})
for (name, _), r, tr in zip(cases.items(), resp['predictions'], truths): print(f"{name}   [truth: {tr}]\n    {r['decision']}  | p_sick {r['p_sick']} | dist {r['unfamiliar_distance']} | top: {r['top3'][0]['condition']} ({r['top3'][0]['p']})")
print('\nbad input ->', service.predict({'instances': [{'image_b64': 'not-an-image'}]}))

Read these as a **demo of the mechanics, not evidence of quality**: five photos prove nothing about accuracy, and the first one is a confident miss. On clean lab photos the confidence table shows the top-class probability is informative
(accuracy rises as the threshold does), but we cannot measure calibration on real field photos because RoCoLe has no PlantVillage-class labels. A real release needs that measured, and a fine-tuned model.

## 12 · Retrospective

In [ ]:
t = pd.Series(timings).rename('seconds').round(0).to_frame(); t['share'] = (t.seconds / t.seconds.sum()).map('{:.0%}'.format)
print(t.to_string()); print(f'\ntotal: {t.seconds.sum():.0f}s on {os.cpu_count()} cores, device: {DEVICE}  (this run; the first run in a fresh checkout also builds the feature caches, roughly 15 more minutes)')

### What we learned about the *problem*
* A lab-trained model's clean-photo accuracy says almost nothing about the field, and the backdrop alone leaks the label.
* **The augmentation that helped was the plain one:** zoomed crops plus photometric jitter. Elaborate synthetic field backgrounds looked more impressive, helped the derived score less, did nothing for a direct healthy-vs-sick head and cost more clean accuracy.
* **How we chose mattered.** We picked the recipe on a development half of the real-world photos and scored a locked half once, split by plant. A synthetic proxy test would have pointed us at the wrong recipe.
* The real-world check is a **triage** signal (healthy vs sick) because crops and diseases don't overlap, on one field, one phone and one annotator, with frozen features.
* A confidence-and-familiarity check lets the app **say "ask an expert"** instead of guessing, including for crops it has never seen.

### What was painful about *doing it this way*
1. **Everything lived in one notebook's memory,** plus ad-hoc cache files. Which data, code and settings produced `cropguard_v1` is only knowable by reading this top to bottom.
2. **Building each augmented feature set is a slow, serial loop** (generate, extract) and the recipe sweep needs ten of them. On all 54k images, or with a fine-tuned network, this is hours.
3. **The sweep and the 16 tuning trials ran one after another** though every run is independent.
4. **One process, one machine.** Fine-tuning the whole network on the full dataset is a throughput problem we cannot show on CPU.
5. **The "pipeline" is Python functions in a notebook.** Re-running it when new field photos arrive means someone remembering to do it.
6. **Serving is a class in a notebook.** A phone app needs a stable endpoint, versioned artifacts and a way to roll back.
7. **Data access depends on live downloads** from third-party sites, which a shared tutorial cannot rely on.

Hold on to that list: it is the reason the next layer of tooling exists.